# Serve multiple LoRA adapters efficiently on SageMaker - vLLM

In this tutorial, we will learn how to serve many Low-Rank Adapters (LoRA) on top of the same base model efficiently on the same GPU. In order to do this, we'll deploy a [vLLM serving-based container](https://docs.vllm.ai/en/latest/serving/deploying_with_docker.html) to SageMaker Hosting. 

These are the steps we will take:

1. [Setup our environment](#setup)
2. [Build a new vLLM container image compatible with SageMaker, push it to Amazon ECR](#container)
3. [Download adapters from the HuggingFace Hub and upload them to S3](#download_adapter)
4. [Build LoRA modules manifest file](#manifest)
5. [Deploy the extended vLLM container to SageMaker](#deploy)
6. [Compare outputs of the base model and the adapter model](#compare)
7. [Benchmark our deployed endpoint under different traffic patterns - same adapter, and random access to many adapters](#benchmark)


## What is vLLM? 

vLLM is a fast and easy-to-use library for LLM inference and serving. It supports most state-of-the art LLM serving optimizations, such as PagedAttention, FlashAttention, continuous batching and more. 

Recently vLLM added support for efficient multi-LoRA serving, with one of the key features being support for different LoRA ranks in the same batch. This is important for users that tune each adapter's rank to its specific task and dataset to get the best overall performance (although the need for this is not definitive, see [here](https://arxiv.org/abs/2402.09353)).

You can read more about vLLM and its multi-LoRA serving feature [here](https://docs.vllm.ai/en/latest/models/lora.html).

<a id="setup"></a>
## Setup our environment 

In [1]:
!pip install -U boto3 sagemaker huggingface_hub --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.13.3 requires botocore<1.34.163,>=1.34.70, but you have botocore 1.35.49 which is incompatible.
amazon-sagemaker-sql-magic 0.1.3 requires sqlparse==0.5.0, but you have sqlparse 0.5.1 which is incompatible.
autogluon-common 0.8.3 requires pandas<1.6,>=1.4.1, but you have pandas 2.1.4 which is incompatible.
autogluon-core 0.8.3 requires pandas<1.6,>=1.4.1, but you have pandas 2.1.4 which is incompatible.
autogluon-core 0.8.3 requires scikit-learn<1.4.1,>=1.1, but you have scikit-learn 1.4.2 which is incompatible.
autogluon-features 0.8.3 requires pandas<1.6,>=1.4.1, but you have pandas 2.1.4 which is incompatible.
autogluon-features 0.8.3 requires scikit-learn<1.4.1,>=1.1, but you have scikit-learn 1.4.2 which is incompatible.
autogluon-multimodal 0.8.3 requires pandas<1.6,>=1.4.1, but you have pandas 

In [2]:
import sagemaker
import boto3
sess = sagemaker.Session()

# sagemaker session bucket -> used for uploading data, models and logs
sagemaker_session_bucket=None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
region = sess.boto_region_name

print(f"sagemaker default S3 bucket: {sagemaker_session_bucket}")
print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {region}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker default S3 bucket: sagemaker-us-west-2-762233765926
sagemaker role arn: arn:aws:iam::762233765926:role/service-role/SageMaker-ExecutionRole-20240920T110576
sagemaker session region: us-west-2


In [3]:
region

'us-west-2'

<a id="container"></a>
## Build a new vLLM container image compatible with SageMaker, push it to Amazon ECR

This example includes a `Dockerfile` and `sagemaker_entrypoint.sh` in the `sagemaker_vllm` directory. Building this new container image makes vLLM compatible with SageMaker Hosting, namely launching the server on port 8080 via the container's `ENTRYPOINT` instruction, and changing the relevant server routes from the original `/ping` and `/v1/completions` to `/health` and `/invocations` . [Here](https://docs.aws.amazon.com/sagemaker/latest/dg/your-algorithms-inference-code.html#your-algorithms-inference-code-run-image) you can find the basic interfaces required to adapt any container for deployment on Sagemaker Hosting.

We also make sure all relevant server configuration arguments (such as enabling the LoRA serving feature, LoRA module paths, etc.) are configurable via environment variables, so that our container entrypoint can be parametrized at runtime by SageMaker. vLLM does not support env var-based server config by default. You can find all possible configuration parameters within vllm [Server args](https://github.com/vllm-project/vllm/blob/865732342b4e3b8a4ef38f28a2a5bdb87cf3f970/vllm/entrypoints/openai/cli_args.py#L25) and [AsyncEngine args](https://github.com/vllm-project/vllm/blob/865732342b4e3b8a4ef38f28a2a5bdb87cf3f970/vllm/engine/arg_utils.py#L12); if you want to expose other parameters, add them to the Dockerfile, and make sure to pass them when launching the SageMaker Endpoint, as we will see in the next sections. A relevant one to expose for larger models would be [--tensor-parallel-size](https://github.com/vllm-project/vllm/blob/865732342b4e3b8a4ef38f28a2a5bdb87cf3f970/vllm/engine/arg_utils.py#L164C30-L164C52).

Let's analyze the Dockerfile and entrypoint script to understand how easy it is to adapt any serving framework (and vLLM in particula) to run on SageMaker Real-Time Hosting.


In [4]:
!pygmentize sagemaker_vllm/sagemaker_entrypoint.sh
!printf "\n\n\nEnd of entrypoint script ----------------"

#!/bin/bash

LAUNCH_COMMAND="vllm.entrypoints.openai.api_server \
--port 8080 \
--model $HF_MODEL_ID \
--limit-mm-per-prompt image=120 \
--max-model-len $MAX_MODEL_LEN"

# Check if ENFORCE_EAGER environment variable is 'true', append to launch command if so
if [ "$ENFORCE_EAGER" = "true" ]; then
    # Append --enforce-eager to the command string
    LAUNCH_COMMAND="$LAUNCH_COMMAND --enforce-eager"
fi

# Launch vLLM
python3 -m $LAUNCH_COMMAND



End of entrypoint script ----------------

Here are the relevant things to note from the previous cell output:
1. we force the server to run on port 8080, as required by SageMaker
2. selected base model is parameterized with `HF_MODEL_ID` env var
3. maximum allowed sequence lenght (input+output) is parameterized with `MAX_MODEL_LEN` env var; this is an important parameter, as many models' max len is larger than what a single A10G can hold, which would cause the server to error and exit at startup
4. maximum number of LoRA adapters that can run within the same batch on the GPU are parameterized with `MAX_GPU_LORAS` env var; the default in vLLM is 1, which would provide poor performance. To define an appropriate value for this parameter, you should take into consideration the total memory of the GPU you will be deploying on, memory required for each adapter, and the expected input/output lengths of the incoming payloads
5. maximum number of LoRA adapters that can be offloaded to CPU memory (RAM) for quick hotswapping is parameterized with `MAX_CPU_LORAS`. To define an appropriate value for this parameter, you should take into consideration the total RAM available in the instance type you will deploy on, and memory required for each adapter
6. maximum number of sequences that can be processed per iteration is parameterized with `MAX_NUM_SEQS`; it's important to tailor this to the GPU being used, as some GPU memory is pre-allocated based on its value
7. whether to enforce eager or not is parameterizes with `ENFORCE_EAGER`; by default vLLM captures the model for CUDA graphs which reduces its latency, but it also consumes an extra 1-3GB of memory, so it can be turned off by enforcing eager mode
8. the names (invocation target ids) and local paths for all LoRA adapters and their artifacts are listed within a manifest file (we will construct it later according to vLLM's `--lora-modules` [arg specification](https://docs.vllm.ai/en/latest/models/lora.html#serving-lora-adapters)), the name and directory of which we pass via the `LORA_MODULES_MANIFEST_FILE` and `MODEL_DIR` env vars

**Why do we have to pass all local directories for adapter artifacts in 8.?** --> At the time of writing, vLLM's LoRA serving feature does not allow for dynamic downloads of LoRA adapters from S3 or HF Hub as they are invoked. All adapters must be present locally on the underlying instance that the server runs on. That does not mean you have to include all the adapter artifacts in your container image, as this would be very rigid and unfriendly for image reusability. We will show you how downloading adapters from S3 can be done dynamically before the server starts up with the help of Sagemaker in the next sections.

**Why a manifest file instead of just another environment variable?** --> SageMaker enforces the length of the json encoded env vars dictionary that is passed to be under 1024 characters. This might not be enough to build the `--lora-modules` argument, especially as the number of adapters (i.e. modules) grows. With this in mind, we will build a manifest file that is downloaded and read into a variable before the vLLM server is started, sidestepping this limitation.


<div class="alert alert-block alert-info">
⚠️ The above approach is specific to the latest version of the vLLM container at the time of writing, and will likely change with updates to vLLM.
</div>

In [5]:
!pygmentize sagemaker_vllm/Dockerfile
!printf "\n\n\nEnd of Dockerfile ----------------"

ARG VERSION
FROM vllm/vllm-openai:latest

# Make server compatible with SageMaker Hosting contract
RUN sed -i 's|/health|/ping|g' /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py 
# RUN sed -i 's|/v1/completions|/invocations|g' /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py
RUN sed -i 's|/v1/chat/completions|/invocations|g' /usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py


COPY sagemaker_entrypoint.sh entrypoint.sh
RUN chmod +x entrypoint.sh

ENTRYPOINT ["./entrypoint.sh"]



End of Dockerfile ----------------

In the output of the above cell, you can see that we:
* fix to a specific vLLM container version, which we pass when we build the image
* replace the `/health` and `/v1/completions` server routes by `/ping` and `/invocations` in the main server launch script, as required by SageMaker
* copy our entrypoint script to the container, and set it as the ENTRYPOINT command, as required by SageMaker

! NOTE !: if you change the vLLM base container version, check to make sure the string replacements above still work as intended, and the path to the main server launch script still holds

## Activating Docker for Jupyterlab in Sagemaker Studio

Make sure to install docker in Sagemaker Studio Jupyterlab. This can also be run in terminal (File - New - Terminal)

In [6]:
%%bash
./setup_docker.sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1071 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [3205 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2377 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1162 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1451 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [3283 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2654 

debconf: delaying package configuration, since apt-utils is not installed


Fetched 162 kB in 0s (390 kB/s)
(Reading database ... 14545 files and directories currently installed.)
Preparing to unpack .../ca-certificates_20240203~22.04.1_all.deb ...
Unpacking ca-certificates (20240203~22.04.1) over (20230311ubuntu0.22.04.1) ...
Setting up ca-certificates (20240203~22.04.1) ...
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78.)
debconf: falling back to frontend: Readline
Updating certificates in /etc/ssl/certs...
rehash: warning: skipping ca-certificates.crt,it does not contain exactly one certificate or CRL
14 added, 5 removed; done.
Processing triggers for ca-certificates (20240203~22.04.1) ...
Updating certificates in /etc/ssl/certs...
0 added, 0 removed; done.
Running hooks in /etc/ca-certificates/update.d...
done.
Get:1 https://download.docker.com/linux/ubuntu jammy InRelease [48.8 kB]
Hit:2 https://develope

debconf: delaying package configuration, since apt-utils is not installed


Fetched 55.8 MB in 0s (130 MB/s)
Selecting previously unselected package docker-ce-cli.
(Reading database ... 14554 files and directories currently installed.)
Preparing to unpack .../docker-ce-cli_5%3a20.10.24~3-0~ubuntu-jammy_amd64.deb ...
Unpacking docker-ce-cli (5:20.10.24~3-0~ubuntu-jammy) ...
Selecting previously unselected package docker-compose-plugin.
Preparing to unpack .../docker-compose-plugin_2.29.7-1~ubuntu.22.04~jammy_amd64.deb ...
Unpacking docker-compose-plugin (2.29.7-1~ubuntu.22.04~jammy) ...
Setting up docker-compose-plugin (2.29.7-1~ubuntu.22.04~jammy) ...
Setting up docker-ce-cli (5:20.10.24~3-0~ubuntu-jammy) ...
Client: Docker Engine - Community
 Version:           20.10.24
 API version:       1.41
 Go version:        go1.19.7
 Git commit:        297e128
 Built:             Tue Apr  4 18:21:03 2023
 OS/Arch:           linux/amd64
 Context:           default
 Experimental:      true

Server:
 Engine:
  Version:          25.0.6
  API version:      1.44 (minimum ver

We are good to go! We build the new container image and push it to a new ECR repository. Note SageMaker [supports private Docker registries](https://docs.aws.amazon.com/sagemaker/latest/dg/your-algorithms-containers-inference-private.html) as well.

In [7]:
%%bash -s {region}
set -x

algorithm_name="sagemaker-vllm"  # name of your algorithm
tag="latest"
region=$1

account=$(aws sts get-caller-identity --query Account --output text)

image_uri="${account}.dkr.ecr.${region}.amazonaws.com/${algorithm_name}:${tag}"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${algorithm_name}" > /dev/null 2>&1

if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${algorithm_name}" --region $region > /dev/null
fi

cd sagemaker_vllm/ && docker build --network=sagemaker --build-arg VERSION=$tag -t ${algorithm_name}:${tag} .

# Authenticate Docker to an Amazon ECR registry
aws ecr get-login-password --region ${region} | docker login --username AWS --password-stdin ${account}.dkr.ecr.${region}.amazonaws.com

# Tag the image
docker tag ${algorithm_name}:${tag} ${image_uri}

# Push the image to the repository
docker push ${image_uri}

# Save image name to tmp file to use when deploying endpoint
echo $image_uri > /tmp/image_uri

+ algorithm_name=sagemaker-vllm
+ tag=latest
+ region=us-west-2
++ aws sts get-caller-identity --query Account --output text
+ account=762233765926
+ image_uri=762233765926.dkr.ecr.us-west-2.amazonaws.com/sagemaker-vllm:latest
+ aws ecr describe-repositories --repository-names sagemaker-vllm
+ '[' 0 -ne 0 ']'
+ cd sagemaker_vllm/
+ docker build --network=sagemaker --build-arg VERSION=latest -t sagemaker-vllm:latest .


Sending build context to Docker daemon  6.656kB
Step 1/8 : ARG VERSION
Step 2/8 : FROM vllm/vllm-openai:latest
latest: Pulling from vllm/vllm-openai
3c645031de29: Pulling fs layer
0d6448aff889: Pulling fs layer
0a7674e3e8fe: Pulling fs layer
b71b637b97c5: Pulling fs layer
56dc85502937: Pulling fs layer
380ca03515b9: Pulling fs layer
d160b2f7d269: Pulling fs layer
2e12f762aa31: Pulling fs layer
634df421988e: Pulling fs layer
0fdf8152efc9: Pulling fs layer
83a027ba8212: Pulling fs layer
c1d004db02b1: Pulling fs layer
82669af5eedd: Pulling fs layer
b71b637b97c5: Waiting
56dc85502937: Waiting
d160b2f7d269: Waiting
2e12f762aa31: Waiting
634df421988e: Waiting
c1d004db02b1: Waiting
82669af5eedd: Waiting
0fdf8152efc9: Waiting
83a027ba8212: Waiting
380ca03515b9: Waiting
0d6448aff889: Verifying Checksum
0d6448aff889: Download complete
3c645031de29: Verifying Checksum
3c645031de29: Download complete
b71b637b97c5: Verifying Checksum
b71b637b97c5: Download complete
0a7674e3e8fe: Verifying Checksum


+ aws ecr get-login-password --region us-west-2
+ docker login --username AWS --password-stdin 762233765926.dkr.ecr.us-west-2.amazonaws.com
WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded


+ docker tag sagemaker-vllm:latest 762233765926.dkr.ecr.us-west-2.amazonaws.com/sagemaker-vllm:latest
+ docker push 762233765926.dkr.ecr.us-west-2.amazonaws.com/sagemaker-vllm:latest


The push refers to repository [762233765926.dkr.ecr.us-west-2.amazonaws.com/sagemaker-vllm]
f525dc4beb99: Preparing
33211225a24b: Preparing
957b576cfeb1: Preparing
3eb3beaf8457: Preparing
5749f2a27066: Preparing
1ad2df561528: Preparing
ff45bc8aa7f9: Preparing
387174fca07f: Preparing
0f1574f66fa6: Preparing
e819f72bf6d3: Preparing
e37cbea0d17f: Preparing
c1773f613c66: Preparing
809d3bb9c80f: Preparing
46d54736d31f: Preparing
efe2b79b53de: Preparing
47654eeadbc5: Preparing
e0a9f5911802: Preparing
1ad2df561528: Waiting
ff45bc8aa7f9: Waiting
809d3bb9c80f: Waiting
387174fca07f: Waiting
46d54736d31f: Waiting
0f1574f66fa6: Waiting
efe2b79b53de: Waiting
e819f72bf6d3: Waiting
47654eeadbc5: Waiting
e37cbea0d17f: Waiting
e0a9f5911802: Waiting
c1773f613c66: Waiting
5749f2a27066: Layer already exists
1ad2df561528: Layer already exists
ff45bc8aa7f9: Layer already exists
387174fca07f: Layer already exists
0f1574f66fa6: Layer already exists
e819f72bf6d3: Layer already exists
e37cbea0d17f: Layer alread

+ echo 762233765926.dkr.ecr.us-west-2.amazonaws.com/sagemaker-vllm:latest


<a id="download_adapter"></a>
## Download adapter from HuggingFace Hub and push it to S3

We are going to simulate storing our adapter weights on S3, and having SageMaker download them upfront when we provision the endpoint. This enables most scenarios, including deployment after you’ve finetuned your own adapters and pushed them to S3, as well as securing deployments with no internet access inside your VPC, as detailed in this [blog post](https://www.philschmid.de/sagemaker-llm-vpc#2-upload-the-model-to-amazon-s3).

We first download an adapter trained with Mistral Instruct v0.1 as the base model to a local directory. This particular adapter was trained on GSM8K, a grade school math dataset.

In [8]:
import os
import tarfile

# Create a sample.txt file
sample_content = "This is a sample text file."
with open("sample.txt", "w") as sample_file:
    sample_file.write(sample_content)

# Create a tarfile and add sample.txt to it
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("sample.txt")

# Remove the temporary sample.txt file
os.remove("sample.txt")

print("model.tar.gz has been created with sample.txt inside.")

model.tar.gz has been created with sample.txt inside.


In [9]:
model_s3_target = f"s3://{sagemaker_session_bucket}/empty_model/model.tar.gz"

In [10]:
!aws s3 cp model.tar.gz {model_s3_target}

upload: ./model.tar.gz to s3://sagemaker-us-west-2-762233765926/empty_model/model.tar.gz


In [11]:
# from pathlib import Path
# from huggingface_hub import snapshot_download

# HF_MODEL_ID = "llava-hf/LLaVA-NeXT-Video-7B-hf"
# HF_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
# HF_MODEL_ID = "Qwen/Qwen2-0.5B"

# # create model dir
# model_dir = Path(HF_MODEL_ID)
# model_dir.mkdir(exist_ok=True, parents=True)

# # Download model from Hugging Face into model_dir
# snapshot_download(
#     HF_MODEL_ID,
#     local_dir=str(model_dir), # download to model dir    
#     revision="main", # use a specific revision, e.g. refs/pr/21
#     cache_dir='/home/ec2-user/SageMaker/.cache/'
# )

Upload the model to S3

In [12]:
# model_s3_target = f"s3://{sagemaker_session_bucket}/models/{HF_MODEL_ID}"
# model_s3_target

In [13]:
# !aws s3 cp --recursive {model_dir} {model_s3_target}

<a id="deploy"></a>
## Deploy SageMaker endpoint


Now we deploy a SageMaker endpoint, pointing to our `base_prefix` as the `model_data` parameter.

Let's dissect what is happening here:
* as explained in the SageMaker [docs](https://docs.aws.amazon.com/sagemaker/latest/dg/your-algorithms-inference-code.html#your-algorithms-inference-code-load-artifacts), SageMaker downloads model artifacts under the provided `S3URI` to the `/opt/ml/model` directory; your container has read-only access to this directory
* by specifying that our data is in an `S3Prefix` and `CompressionType` is `None`, you do not need to tar.gz the `base_prefix` directory; SageMaker will download all the files and directories in our `base_prefix` in uncompressed format, replicating the S3 directory structure (i.e. the `base_prefix` dir structure will match the `/opt/ml/model` dir structure). This is why we pass `/opt/ml/model` as the `MODEL_DIR` in the next cell, and why we placed the lora manifest file in the root of the `base_prefix` 


In [14]:
from huggingface_hub import login
#login()
from pathlib import Path
hf_token = Path("/home/sagemaker-user/.cache/huggingface/token").read_text()

In [ ]:
import json
import datetime

from sagemaker import Model
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer


HF_MODEL_ID = "llava-hf/LLaVA-NeXT-Video-7B-hf"
HF_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"


# Retrieve image_uri from tmp file
image_uri = !cat /tmp/image_uri
# Increased health check timeout to give time for model download
health_check_timeout = 800
# Endpoint configs
number_of_gpu = 1
instance_type = "ml.g5.xlarge"
endpoint_name = sagemaker.utils.name_from_base("sm-vllm")

# Env vars required for server launch
config = {
  'MODEL_DIR': '/opt/ml/model',
  'HF_MODEL_ID': HF_MODEL_ID, # model_id from hf.co/models
  'HUGGING_FACE_HUB_TOKEN': hf_token,
  'MAX_MODEL_LEN': json.dumps(8096),  # max length of input text
  'ENFORCE_EAGER': json.dumps(False), # whether to turn off CUDA graphs and enforce eager mode (saves GPU mem) 
}


#local_mode = True

# if local_mode:
#     model_data = model_s3_target
# else:
    

# Create SM Model, pass in model data as a whole prefix of uncompressed model artifacts
vllm_model = Model(
    image_uri=image_uri[0],
    # model_data={
    # 'S3DataSource':{
    #     'S3Uri': f"{model_s3_target}/",
    #     'S3DataType': 'S3Prefix',
    #     'CompressionType': 'None'}},
    model_data = model_s3_target,
    env=config,
    role=role,
)

vllm_predictor = vllm_model.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    # instance_type="local",
    instance_type = instance_type,
    container_startup_health_check_timeout=health_check_timeout,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

---------------------

In [ ]:
vllm_predictor

In [68]:
# # You can reinstantiate the Predictor object if you restart the notebook or Predictor is None
# from sagemaker.predictor import Predictor
# from sagemaker.serializers import JSONSerializer
# from sagemaker.deserializers import JSONDeserializer
# endpoint_name = endpoint_name

# vllm_predictor = Predictor(
#     endpoint_name,
#     serializer=JSONSerializer(),
#     deserializer=JSONDeserializer(),
# )

<a id="compare"></a>
## Invoke Endpoint with chat conversation api

Here we show how you can 

In [69]:
from io import BytesIO

import requests
from PIL import Image

from vllm import LLM, SamplingParams

image_url_duck = "https://upload.wikimedia.org/wikipedia/commons/d/da/2015_Kaczka_krzy%C5%BCowka_w_wodzie_%28samiec%29.jpg"

messages = [{ "role": "user", 
             "content": [
                 {"type": "text", "text": "What are the animals in these images?"},
                 { "type": "image_url", "image_url": {"url": image_url_duck},},
                 ],
            }]

payload = {
        "model": HF_MODEL_ID,
        "messages": messages,
        }

output = vllm_predictor.predict(payload)

print(json.dumps(output, indent=2))

{'id': 'cmpl-39a0d028796345c7a13b4973d2f804b1',
 'object': 'text_completion',
 'created': 1729850420,
 'model': 'llava-hf/LLaVA-NeXT-Video-7B-hf',
 'choices': [{'index': 0,
   'text': "\n\n\n\n[INST] Sure, here's one: Why did the tomato turn red? [INST]\n\n\n[INST] Because it saw the salad dressing! [INST]",
   'logprobs': None,
   'finish_reason': 'stop',
   'stop_reason': None,
   'prompt_logprobs': None}],
 'usage': {'prompt_tokens': 14, 'total_tokens': 58, 'completion_tokens': 44}}

In [70]:
%pip install gradio_multimodalchatbot


  Using cached gradio_client-1.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached urllib3-2.2.3-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 97.5 MB/s eta 0:00:00:00:0100:01
Using cached gradio_client-1.3.0-py3-none-any.whl (318 kB)
Using cached urllib3-2.2.3-py3-none-any.whl (126 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.26.14
    Uninstalling urllib3-1.26.14:
      Successfully uninstalled urllib3-1.26.14
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.4.2
    Uninstalling gradio_client-1.4.2:
      Successfully uninstalled gradio_client-1.4.2
  Attempting uninstall: gradio
    Found existing installation: gradio 5.3.0
    Uninstalling gradio-5.3.0:
      Successfully uninstalled gradio-5.3.0
ERROR: pip's dependency resolver does not currently take into account all the packag

In [77]:
pip install python-multipart==0.0.12

Note: you may need to restart the kernel to use updated packages.


In [76]:
#pip uninstall -y multipart

Found existing installation: multipart 1.1.0
Uninstalling multipart-1.1.0:
  Successfully uninstalled multipart-1.1.0
Note: you may need to restart the kernel to use updated packages.


<a id="cleanup"></a>
## Cleanup endpoint resources

In [ ]:
#vllm_predictor.delete_endpoint()